# Оценка релевантности организаций запросам на Яндекс.Картах с помощью LLM-агента

<img src="https://sun9-65.userapi.com/impg/N4y2cxlL7PauAs82tBNFOUAiNctFICWDy4Mbiw/Jiz1fb7NLWU.jpg?size=1080x1080&quality=95&sign=df2786058624d9ccac3ede4d5d056e2f&type=album" width="500" height="500" />


## Описание и загрузка данных

In [ ]:
import requests

public_url = "https://disk.yandex.ru/d/6d5hFHvpAZjQdw"  # твоя публичная ссылка
api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

resp = requests.get(api_url, params={"public_key": public_url})
resp.raise_for_status()
download_url = resp.json()["href"]  # это уже прямая ссылка на файл

dest = "/content/data.jsonl"
with requests.get(download_url, stream=True) as r:
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)


In [ ]:
import json
import pandas as pd

records = []
with open("/content/data.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if i == 2659:      # пропускаем битую строку
            continue
        try:
            obj = json.loads(line)
            records.append(obj)
        except Exception as e:
            print("ещё битая строка:", i, e)

data = pd.DataFrame(records)


In [ ]:
data['relevance'].unique()

array([1. , 0. , 0.1])

In [ ]:
data['relevance'].value_counts()

,count
relevance,
1.0,15881
0.0,14509
0.1,4703


Здесь 1.0 соответствует оценке RELEVANT_PLUS, 0.1 -- оценке RELEVANT_MINUS, 0.0 -- оценке IRRELEVANT.

Ваша задача -- построить LLM-агента, который будет предсказывать релевантность.

Выделим данные для оценки качества агента. Запуск агента -- это тяжелая и потенциально дорогая операция. Поэтому eval-множество имеет размер 500. Также для простоты из eval-множества выкинуты данные с оценкой RELEVANT_MINUS. Тем не менее, вы можете использовать такие примеры для подачи примеров агенту.

**ОБРАТИТЕ ВНИМАНИЕ, ЧТО В EVAL-ДАННЫЕ НЕЛЬЗЯ ПОДГЛЯДЫВАТЬ ДЛЯ КАЛИБРОВКИ АГЕНТА!!! ДЛЯ ЭТОГО ЕСТЬ ОБУЧАЮЩИЕ ДАННЫЕ**

В качестве метрики качества мы будем использовать обычную ACCURACY, поскольку классы сбалансированы.

In [ ]:
train_data = data[570:]
eval_data = data[:570]
eval_data = eval_data[eval_data["relevance"] != 0.1]
eval_data

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized
0,сигары,"Москва, Дубравная улица, 34/29",Tabaccos; Магазин Tabaccos; Табаккос,Магазин табака и курительных принадлежностей,1263329400,None,1.0,"Организация занимается продажей табака, курите..."
1,кальянная спб мероприятия,"Санкт-Петербург, Большой проспект Петроградско...",PioNero; Pionero; Пицца Паста бар; Pio Nero; P...,Кафе,228111266197,PioNero предлагает разнообразные блюда итальян...,0.0,"Организация PioNero — это кафе, бар и ресторан..."
2,Эпиляция,"Московская область, Одинцово, улица Маршала Жу...",MaxiLife; Центр красоты и здоровья MaxiLife; Ц...,Стоматологическая клиника,1247255817,"Стоматологическая клиника, массажный салон и к...",1.0,"Организация занимается стоматологическими, кос..."
4,стиральных машин,"Москва, улица Обручева, 34/63",М.Видео; M Video; M. Видео; M.Видео; Mvideo; М...,Магазин бытовой техники,1074529324,М.Видео предлагает широкий ассортимент бытовой...,1.0,Организация занимается продажей бытовой техник...
5,сеть быстрого питания,"Санкт-Петербург, 1-я Красноармейская улица, 15",Rostic's; KFC; Ресторан быстрого питания KFC,Быстрое питание,1219173871,Rostic's предлагает различные наборы быстрого ...,1.0,"Организация занимается быстрым питанием, предо..."
...,...,...,...,...,...,...,...,...
561,наращивание ресниц,"Саратов, улица имени А.С. Пушкина, 1",Сила; Sila; Beauty brow; Студия бровей Beauty ...,Салон красоты,236976975812,Салон красоты «Сила» предлагает услуги по уход...,1.0,Организация «Сила» занимается предоставлением ...
565,игры,"Москва, Щёлковское шоссе, 79, корп. 1",YouPlay; YouPlay КиберКлуб,Компьютерный клуб,109673025161,YouPlay КиберКлуб предлагает услуги по игре на...,0.0,Организация занимается предоставлением услуг к...
566,домашний интернет в курске что подключить отзы...,"Курск, Садовая улица, 5",Цифровой канал; Digital Channel; DChannel; ЦК;...,Телекоммуникационная компания,1737991898,None,0.0,None
567,гостиница волгодонск сауна номер телефона,"Ростовская область, городской округ Волгодонск...",Поплавок; Poplavok,"База , дом отдыха",147783493467,"Предлагает размещение в различных типах жилья,...",0.0,Организация «Поплавок» предлагает услуги базы ...


In [ ]:
eval_data.to_excel("eval_data.xlsx")

# Агент


## Агент с тулзой

In [ ]:
!pip install -q langchain langgraph langchain_openai langchain_core langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import tool
import os
import time, uuid
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
tavily_log_desc = """
Use this tool ONLY when the organization card
does not contain enough information to determine strict relevance.

This tool is used to VERIFY missing attributes explicitly mentioned
in the user query (e.g. age limits, accommodation, cheap/expensive,
specific services, brands, 24/7, veranda, live music, exact location).

Build the search query using:
organization name + city/address + the missing attribute.

If the required attribute cannot be confirmed from search results,
assume the organization is NOT relevant.

Do NOT guess. Use the tool only to confirm facts.
"""

In [ ]:
tavily_raw = TavilySearchResults(
    max_results=10,
    include_answer=True,
    include_raw_content=False
)

LOG_PATH = "tavily_log.jsonl"

def log_event(obj: dict):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

#@tool(description=tavily_log_desc)
def tavily_logged(query: str) -> dict:
    f"""{TavilySearchResults.__doc__}"""
    t0 = time.time()
    run_id = str(uuid.uuid4())

    log_event({
        "type": "tool_start",
        "run_id": run_id,
        "tool": "tavily_search_results_json",
        "query": query,
        "ts": t0,
    })

    out = tavily_raw.invoke({"query": query})

    dt = time.time() - t0
    # out обычно dict с "answer" и "results"
    answer = out.get("answer") if isinstance(out, dict) else None
    results = out.get("results") if isinstance(out, dict) else None

    log_event({
        "type": "tool_end",
        "run_id": run_id,
        "tool": "tavily_search_results_json",
        "query": query,
        "latency_sec": dt,
        "answer": answer,
        "top_results": (results[:3] if isinstance(results, list) else None),
    })
    return out



tools = [tavily_logged]


In [ ]:
open_router_model_name = "nousresearch/hermes-3-llama-3.1-405b:free"

llm = ChatOpenAI(
    model=open_router_model_name,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0.3,
    max_tokens=1024,
    timeout=180,
    max_retries=5)#.bind_tools(tools)


In [ ]:
def make_example(example: pd.Series, is_train: bool = False):
  ans=f'Запрос для оценки релевантности: {example['Text']}.\n'
  ans+='Информация об организации:\n'
  ans+=f'Адрес: {example['address']}.\n'
  ans+=f'Название: {example['address']}.\n'
  if example['normalized_main_rubric_name_ru']: ans+=f'Описание ассортимента: {example['normalized_main_rubric_name_ru'][:1000]}.\n'
  if example['reviews_summarized'] : ans+=f'Краткое содержание отзывов: {example['reviews_summarized'][:1000]}\n'
  if is_train: ans+=f'Оценка релевантности: {int(example['relevance'])}'
  return ans

Возможно попробовтаь сначала суммаризировать reviews_summarized

In [ ]:
search_tool_name = f'Инструмент для веб-поиска (tavily_logged.name)'
tavily_logged.name

'tavily_logged'

In [ ]:
train_few_shot_ids = [570, 571, 586, 581, 590, 983]

SYSTEM_PROMPT = f'''
Ты - ассистент, который оценивает, строгую релевантность организаций на картах широким рубричным запросам.\n
Примеры рубричных запросов: "ресторан с верандой", "романтичный джаз-бар".\n
Оцени релевантность организации запросу одним числом: 0 или 1. Возможные ответы:\n
1 - Организация полностью релевантна запросу.\n
0 - Организация НЕ релевантна запросу.\n
Примеры:\n
{make_example(train_data.loc[570],is_train=True)}\n\n
{make_example(train_data.loc[571],is_train=True)}\n\n
{make_example(train_data.loc[586],is_train=True)}\n\n
{make_example(train_data.loc[581],is_train=True)}\n\n
{make_example(train_data.loc[590],is_train=True)}\n\n
{make_example(train_data.loc[983],is_train=True)}\n\n
ВАЖНО : В рубричном запросе очень важны детали, если деталь не верна, то запрос не релевантен.\n
Если уверенность в ответе более 0.8 - то есть данных достаточно для уверенного точного ответа
то — дай в ответе число: 0 или 1.\n
Если уверенность в ответе менее 0.8 - то есть данных недостаточно — НЕ ВОЗВРАЩАЙ ЧИСЛО,
а ОБЯЗАТЕЛЬНО вызови {search_tool_name} с уточняющим запросом по конкретным деталям,
которых нет в представленной информации
Пример уточняющего запроса для уточнения, можно ли в Ингосстрах застраховать именно автомобиль:\n
Страхование автомобиля, застраховать машину в Ингосстрах по адресу Вологодская область,
рабочий посёлок Чагода, улица Кирова, 5Б.\n
Возможный шаблон запроса: "Есть ли <ДЕТАЛЬ> в <НАЗВАНИЕ ОРГАНИЗАЦИИ> по адресу <АДРЕС> ?"'''


In [ ]:
SYSTEM_PROMPT = f"""
Ты — ассистент, который оценивает СТРОГУЮ релевантность организации рубричному запросу пользователя.

ВАЖНО: Ты ДОЛЖЕН отвечать ТОЛЬКО валидным JSON. Никакого текста вне JSON.
Запрещено: Markdown, тройные кавычки, комментарии, пояснения вне JSON, лишние поля.

Задача:
Вернуть метку релевантности:
- 1: организация полностью релевантна запросу
- 0: организация нерелевантна запросу

Критерий строгой релевантности:
Если любая важная деталь запроса не выполняется или противоречит карточке — ответ 0.

Инструменты:
У тебя есть инструмент веб-поиска: {search_tool_name}.
Используй его ТОЛЬКО чтобы проверить отсутствующие или двусмысленные детали.

=== Few-shot примеры ===
{make_example(train_data.loc[570], is_train=True)}

{make_example(train_data.loc[571], is_train=True)}

{make_example(train_data.loc[586], is_train=True)}

{make_example(train_data.loc[581], is_train=True)}

{make_example(train_data.loc[590], is_train=True)}

{make_example(train_data.loc[983], is_train=True)}

=== Алгоритм ===
1) Определи ожидаемый тип организации из запроса.
2) Извлеки важные детали запроса (обязательные атрибуты), включая:
   - тип организации/услуги/товара
   - география (город/район/адрес), если указано
   - режим работы (24/7, праздники), если указано
   - конкретная специализация/бренд/модель/ключевое слово, если указано
   - субъективные требования (например "романтичный ресторан") — это важно; фастфуд/шаверма/постамат нерелевантны.
3) Проверь, подтверждаются ли детали карточкой организации (название/рубрика/описания/отзывы/адрес).
4) Оцени уверенность в финальном решении:
   - если confidence >= 0.80: поиск НЕ нужен, верни label 0/1.
   - если confidence < 0.80: поиск НУЖЕН. В этом случае:
     a) label ДОЛЖЕН быть null
     b) needs_search = true
     c) сформируй 1–2 поисковых запроса (search_queries), каждый максимально конкретный
     d) ОБЯЗАТЕЛЬНО вызови {search_tool_name} (tool-call) с query из search_queries[0]
5) После того как ты получишь ToolMessage с результатами поиска:
   - занеси их в поле search_results
   - пересчитай confidence
   - верни финальный label 0/1

=== Формирование поискового запроса (search_queries) ===
Каждый запрос должен быть конкретным и проверять ровно одну отсутствующую деталь.
Шаблон:
"<название организации> <город или адрес> <отсутствующая деталь>"
или вопрос:
"Есть ли <деталь> в <название> по адресу <адрес>?"

=== Строгий формат ответа (JSON) ===
Ты ВСЕГДА возвращаешь объект JSON С РОВНО этими ключами (в таком порядке):
1. version (строка) — всегда "1.0"
2. label (число 0 или 1, либо null)
3. confidence (число от 0.0 до 1.0)
4. needs_search (true/false)
5. missing_details (массив строк)
6. search_queries (массив строк)
7. search_results (массив объектов)
8. rationale (строка, 1–3 предложения)
9. decision_rules_hit (массив строк)

Правила:
- Если needs_search=false: label ОБЯЗАН быть 0 или 1; search_queries может быть []; search_results должен быть [].
- Если needs_search=true: label ОБЯЗАН быть null; search_queries должен содержать >=1 строку; search_results должен быть [] ДО получения ToolMessage.
- После получения ToolMessage: needs_search=false, label=0/1, search_results заполнен.

Формат search_results:
Массив объектов вида:
{{
  "query": "<строка запроса>",
  "answer": "<краткий ответ Tavily или пустая строка>",
  "top_results": [
    {{"title":"...", "url":"...", "content":"..."}},
    ...
  ]
}}

Ограничения:
- Никаких дополнительных ключей.
- Никаких переносов и текста вне JSON.
- Всегда валидный JSON (двойные кавычки, без trailing comma).
- rationale — НЕ более 350 символов.

СЕЙЧАС ВЫПОЛНИ ЗАДАЧУ.
"""


In [ ]:
global_debug = []

def should_continue(state: MessagesState):
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

def gather_data(state: MessagesState):

    messages = state["messages"]
    system_message = (SystemMessage(content=SYSTEM_PROMPT))
    response = llm.invoke([system_message] + messages)
    # debug info
    debug = (json.dumps(response.tool_calls, indent=2,ensure_ascii=False),
    json.dumps(response.content, indent=2,ensure_ascii=False))
    print(debug, flush=True)
    global_debug.append(debug)
    return {"messages" : [response]}

tool_node = ToolNode(tools)
workflow = StateGraph(MessagesState)

workflow.add_node("gather_data", gather_data)
workflow.add_node("tools", tool_node)

workflow.add_conditional_edges("gather_data",should_continue,["tools",END])
workflow.add_edge("tools", "gather_data")

workflow.set_entry_point("gather_data")
graph = workflow.compile()

In [ ]:
prompt = make_example(train_data.loc[574])

input_messages = [HumanMessage(prompt)]
output = graph.invoke({"messages": input_messages})

('[]', '"{\\n  \\"version\\": \\"1.0\\",\\n  \\"label\\": null,\\n  \\"confidence\\": 0.6,\\n  \\"needs_search\\": true,\\n  \\"missing_details\\": [\\"Возрастные ограничения\\", \\"Наличие проживания\\"],\\n  \\"search_queries\\": [\\"Школа танцев Невский проспект 35В Санкт-Петербург возрастные ограничения\\", \\"Школа танцев Невский проспект 35В Санкт-Петербург наличие проживания\\"],\\n  \\"search_results\\": [],\\n  \\"rationale\\": \\"Описание и отзывы подтверждают, что это балетная школа в Санкт-Петербурге. Однако не хватает информации о возрасте учащихся и наличии проживания, чтобы полностью соответствовать запросу.\\",\\n  \\"decision_rules_hit\\": []\\n}"')


In [ ]:
print(output['messages'][1].content)

{
  "version": "1.0",
  "label": null,
  "confidence": 0.6,
  "needs_search": true,
  "missing_details": ["Возрастные ограничения", "Наличие проживания"],
  "search_queries": ["Школа танцев Невский проспект 35В Санкт-Петербург возрастные ограничения", "Школа танцев Невский проспект 35В Санкт-Петербург наличие проживания"],
  "search_results": [],
  "rationale": "Описание и отзывы подтверждают, что это балетная школа в Санкт-Петербурге. Однако не хватает информации о возрасте учащихся и наличии проживания, чтобы полностью соответствовать запросу.",
  "decision_rules_hit": []
}


In [ ]:
def parse_model_json(text: str) -> dict:
    try:
        obj = json.loads(text)
        if isinstance(obj, str):
            obj = json.loads(obj)
        return obj
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON from model: {e}")

{'version': '1.0',
 'label': None,
 'confidence': 0.6,
 'needs_search': True,
 'missing_details': ['Возрастные ограничения', 'Наличие проживания'],
 'search_queries': ['Школа танцев Невский проспект 35В Санкт-Петербург возрастные ограничения',
  'Школа танцев Невский проспект 35В Санкт-Петербург наличие проживания'],
 'search_results': [],
 'rationale': 'Описание и отзывы подтверждают, что это балетная школа в Санкт-Петербурге. Однако не хватает информации о возрасте учащихся и наличии проживания, чтобы полностью соответствовать запросу.',
 'decision_rules_hit': []}

In [ ]:
data = parse_model_json(output['messages'][1].content)
data

{'version': '1.0',
 'label': None,
 'confidence': 0.6,
 'needs_search': True,
 'missing_details': ['Возрастные ограничения', 'Наличие проживания'],
 'search_queries': ['Школа танцев Невский проспект 35В Санкт-Петербург возрастные ограничения',
  'Школа танцев Невский проспект 35В Санкт-Петербург наличие проживания'],
 'search_results': [],
 'rationale': 'Описание и отзывы подтверждают, что это балетная школа в Санкт-Петербурге. Однако не хватает информации о возрасте учащихся и наличии проживания, чтобы полностью соответствовать запросу.',
 'decision_rules_hit': []}